# Module 4: Feature Engineering

This notebook is a direct archival split from `feature_engineering_and_model_development.ipynb`.

- The original feature-engineering code and stored cell outputs are preserved.
- It does not contain feature selection, outlier handling, or model development.
- For the current agreed workflow, **do not rerun this notebook**. The group-approved downstream handoff is the separately supplied `data_reclean_after_fe.csv`.


In [1]:
import io
import os
import re
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_samples,
    silhouette_score,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
KEY_COLS = ["id_student", "code_module", "code_presentation"]
OUTPUT_DIR = Path("artifacts")
REPORT_DIR = Path("reports")
FIGURE_DIR = Path("figures")

for path in [OUTPUT_DIR, REPORT_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DRIVE_FILE_URL = "https://drive.google.com/file/d/1eFIYrXSrVlUaA7Pvzu1nP0H-v01NFNQe/view?usp=sharing"
LOCAL_CANDIDATES = [
    "merged_student_behavior.csv",
    "merged_student_behavior.xlsx",
    "shared_data.csv",
    "shared_data.xlsx",
    "data/processed/merged_student_behavior.csv",
    "data/processed/merged_student_behavior.xlsx",
]

VARIANCE_THRESHOLD = 0.01
CORRELATION_THRESHOLD = 0.95
REMOVE_OUTLIERS_FOR_MODEL = True
OUTLIER_CONTAMINATION = 0.05
K_RANGE = range(2, 11)

In [2]:
def read_shared_data():
    for path in LOCAL_CANDIDATES:
        if Path(path).exists():
            if path.lower().endswith(".csv"):
                return pd.read_csv(path)
            return pd.read_excel(path)

    import requests

    match = re.search(r"/d/([^/]+)", DRIVE_FILE_URL)
    if not match:
        raise ValueError("Invalid Google Drive URL.")

    file_id = match.group(1)
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    content = response.content
    try:
        return pd.read_csv(io.BytesIO(content))
    except Exception:
        return pd.read_excel(io.BytesIO(content))


shared_df = read_shared_data()

print("Shared data loaded.")
print("Shape:", shared_df.shape)
print("Unique students:", shared_df["id_student"].nunique() if "id_student" in shared_df.columns else "N/A")
display(shared_df.head())

Shared data loaded.
Shape: (32593, 48)
Unique students: 28785


,id_student,code_module,code_presentation,final_result,login_weekly,dominant_login_time,video_completion_rate,days_to_first_activity,forum_posts_count,submission_timeliness_days,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,date_registration,date_unregistration,vle_total_clicks,vle_active_days,vle_distinct_sites,vle_clicks_dataplus,vle_clicks_dualpane,vle_clicks_externalquiz,vle_clicks_folder,vle_clicks_forumng,vle_clicks_glossary,vle_clicks_homepage,vle_clicks_htmlactivity,vle_clicks_oucollaborate,vle_clicks_oucontent,vle_clicks_ouelluminate,vle_clicks_ouwiki,vle_clicks_page,vle_clicks_questionnaire,vle_clicks_quiz,vle_clicks_repeatactivity,vle_clicks_resource,vle_clicks_sharedsubpage,vle_clicks_subpage,vle_clicks_url,assess_n_submitted,assess_mean_score,assess_n_banked,assess_mean_days_before_deadline,assess_n_late
0,11391,AAA,2013J,Pass,2,Afternoon,0.934,9.0,5,-0.2,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,-159.0,NaN,934.0,40.0,55.0,0.0,0.0,0.0,0.0,193.0,0.0,138.0,0.0,0.0,553.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0,0.0,32.0,5.0,5.0,82.0,0.0,1.8,0.0
1,28400,AAA,2013J,Pass,6,Evening,0.879,2.0,0,-1.5,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,-53.0,NaN,1435.0,80.0,84.0,10.0,0.0,0.0,0.0,417.0,0.0,324.0,0.0,0.0,537.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,0.0,87.0,48.0,5.0,66.4,0.0,0.0,2.0
2,30268,AAA,2013J,Withdrawn,0,Evening,0.522,NaN,0,3.4,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,-92.0,12.0,281.0,12.0,22.0,0.0,0.0,0.0,0.0,126.0,0.0,59.0,0.0,0.0,66.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,22.0,4.0,NaN,NaN,NaN,NaN,NaN
3,31604,AAA,2013J,Pass,8,Evening,0.521,3.0,2,2.2,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,-52.0,NaN,2158.0,123.0,82.0,2.0,0.0,0.0,0.0,634.0,1.0,432.0,0.0,0.0,836.0,0.0,0.0,0.0,0.0,0.0,0.0,19.0,0.0,144.0,90.0,5.0,76.0,0.0,2.0,0.0
4,32885,AAA,2013J,Pass,0,Afternoon,0.240,7.0,2,-0.4,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,-176.0,NaN,1034.0,70.0,66.0,0.0,0.0,0.0,0.0,194.0,4.0,204.0,0.0,0.0,494.0,0.0,0.0,0.0,0.0,0.0,0.0,45.0,0.0,79.0,14.0,5.0,54.4,0.0,-11.4,5.0


In [3]:
required_base_cols = [
    "id_student",
    "code_module",
    "code_presentation",
    "final_result",
    "login_weekly",
    "dominant_login_time",
    "video_completion_rate",
    "days_to_first_activity",
    "forum_posts_count",
    "submission_timeliness_days",
]

missing_required = [c for c in required_base_cols if c not in shared_df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

raw_df = shared_df.copy()
before_rows = len(raw_df)
df = raw_df.drop_duplicates(subset=KEY_COLS).reset_index(drop=True)
after_rows = len(df)

missing_report = (
    df.isna()
    .mean()
    .mul(100)
    .round(2)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "column"})
    .sort_values("missing_pct", ascending=False)
)

print("Rows before de-duplication:", before_rows)
print("Rows after de-duplication by learner-course-presentation:", after_rows)
print("Duplicate rows removed:", before_rows - after_rows)
print("Observation unit:", " + ".join(KEY_COLS))
display(missing_report.head(15))
display(df.describe(include="all").T)

Rows before de-duplication: 32593
Rows after de-duplication by learner-course-presentation: 32593
Duplicate rows removed: 0
Observation unit: id_student + code_module + code_presentation


,column,missing_pct
19,date_unregistration,69.10
44,assess_mean_score,20.78
43,assess_n_submitted,20.71
46,assess_mean_days_before_deadline,20.71
47,assess_n_late,20.71
45,assess_n_banked,20.71
7,days_to_first_activity,13.91
25,vle_clicks_externalquiz,10.32
22,vle_distinct_sites,10.32
21,vle_active_days,10.32


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_student,32593.0,NaN,NaN,NaN,706687.669131,549167.313855,3733.0,508573.0,590310.0,644453.0,2716795.0
code_module,32593,7,BBB,7909,NaN,NaN,NaN,NaN,NaN,NaN,NaN
code_presentation,32593,4,2014J,11260,NaN,NaN,NaN,NaN,NaN,NaN,NaN
final_result,32593,4,Pass,12361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
login_weekly,32593.0,NaN,NaN,NaN,2.616635,3.479342,0.0,0.0,1.0,4.0,24.0
dominant_login_time,32593,4,Afternoon,11390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
video_completion_rate,32593.0,NaN,NaN,NaN,0.444531,0.359951,0.0,0.082,0.388,0.808,1.0
days_to_first_activity,28058.0,NaN,NaN,NaN,10.498396,12.466346,0.0,2.0,6.0,14.0,90.0
forum_posts_count,32593.0,NaN,NaN,NaN,1.449514,2.506308,0.0,0.0,0.0,2.0,19.0
submission_timeliness_days,32593.0,NaN,NaN,NaN,0.362056,3.172153,-11.6,-1.8,0.4,2.6,12.2


In [4]:
def safe_divide(a, b):
    return a / b.replace(0, np.nan)


def safe_zscore(series):
    series = pd.Series(series)
    std = series.std()
    if std == 0 or pd.isna(std):
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.mean()) / std


def first_existing(columns, candidates):
    return [c for c in candidates if c in columns]


def build_feature_table(data):
    work = data.copy()

    numeric_fill_zero = [
        c for c in work.columns
        if c.startswith("vle_") or c.startswith("assess_")
    ]
    for col in numeric_fill_zero:
        if pd.api.types.is_numeric_dtype(work[col]):
            work[col] = work[col].fillna(0)

    work["never_started"] = work["days_to_first_activity"].isna().astype(int)
    fallback_first_day = work["days_to_first_activity"].max()
    if pd.isna(fallback_first_day):
        fallback_first_day = 90
    work["days_to_first_filled"] = work["days_to_first_activity"].fillna(fallback_first_day)

    work["missed_submission"] = (
        (work["submission_timeliness_days"] > 7) |
        (work.get("assess_n_submitted", pd.Series(0, index=work.index)).fillna(0) == 0)
    ).astype(int)

    work["late_submitter"] = (work["submission_timeliness_days"] > 7).astype(int)
    work["prompt_starter"] = (
        work["days_to_first_filled"] <= work["days_to_first_filled"].median()
    ).astype(int)

    work["early_start_score"] = 1 / (1 + work["days_to_first_filled"])
    work["frequency_score"] = safe_zscore(work["login_weekly"]) + safe_zscore(work["forum_posts_count"])
    work["intensity_score"] = (
        safe_zscore(work["video_completion_rate"]) +
        safe_zscore(-work["submission_timeliness_days"])
    )
    work["forum_per_login"] = work["forum_posts_count"] / (work["login_weekly"] + 1)
    work["completion_x_frequency"] = work["video_completion_rate"] * work["login_weekly"]
    work["procrastination_score"] = (
        safe_zscore(work["days_to_first_filled"]) +
        safe_zscore(work["submission_timeliness_days"])
    )

    work["low_engagement_flag"] = (
        (work["login_weekly"] <= work["login_weekly"].quantile(0.25)) &
        (work["video_completion_rate"] <= work["video_completion_rate"].quantile(0.25))
    ).astype(int)

    work["engagement_breadth"] = (
        (work["login_weekly"] > work["login_weekly"].median()).astype(int) +
        (work["video_completion_rate"] > work["video_completion_rate"].median()).astype(int) +
        (work["forum_posts_count"] > work["forum_posts_count"].median()).astype(int) +
        (work["submission_timeliness_days"] < work["submission_timeliness_days"].median()).astype(int)
    )

    if "date_registration" in work.columns:
        work["registration_delay_days"] = work["date_registration"].fillna(work["date_registration"].median())
        work["registered_before_start"] = (work["registration_delay_days"] < 0).astype(int)
    else:
        work["registration_delay_days"] = 0
        work["registered_before_start"] = 0

    if "date_unregistration" in work.columns:
        work["is_unregistered"] = work["date_unregistration"].notna().astype(int)
    else:
        work["is_unregistered"] = 0

    if "vle_total_clicks" in work.columns:
        work["log_vle_total_clicks"] = np.log1p(work["vle_total_clicks"])
    if "vle_active_days" in work.columns:
        work["log_vle_active_days"] = np.log1p(work["vle_active_days"])
    if "vle_distinct_sites" in work.columns:
        work["log_vle_distinct_sites"] = np.log1p(work["vle_distinct_sites"])

    if {"vle_total_clicks", "vle_active_days"}.issubset(work.columns):
        work["clicks_per_active_day"] = safe_divide(work["vle_total_clicks"], work["vle_active_days"]).fillna(0)
        work["log_clicks_per_active_day"] = np.log1p(work["clicks_per_active_day"])

    vle_click_cols = [c for c in work.columns if c.startswith("vle_clicks_")]
    if vle_click_cols:
        total = work[vle_click_cols].sum(axis=1).replace(0, np.nan)
        for col in vle_click_cols:
            work[f"{col}_ratio"] = (work[col] / total).fillna(0)
        work["vle_activity_type_diversity"] = work[vle_click_cols].gt(0).sum(axis=1)

    if "assess_n_submitted" in work.columns:
        total_possible = (
            work.groupby(["code_module", "code_presentation"])["assess_n_submitted"]
            .transform("max")
            .replace(0, np.nan)
        )
        work["assessment_completion_rate"] = (work["assess_n_submitted"] / total_possible).fillna(0)
    else:
        work["assessment_completion_rate"] = 0

    if {"assess_n_late", "assess_n_submitted"}.issubset(work.columns):
        work["late_assessment_rate"] = safe_divide(work["assess_n_late"], work["assess_n_submitted"]).fillna(0)
    else:
        work["late_assessment_rate"] = work["late_submitter"]

    if "assess_mean_days_before_deadline" in work.columns:
        work["assignment_timeliness_score"] = safe_zscore(work["assess_mean_days_before_deadline"])
    else:
        work["assignment_timeliness_score"] = safe_zscore(-work["submission_timeliness_days"])

    login_dummies = pd.get_dummies(work["dominant_login_time"], prefix="login_time").astype(int)
    work = pd.concat([work, login_dummies], axis=1)

    return work


feature_df = build_feature_table(df)

display(feature_df.head())
print("Feature table shape:", feature_df.shape)

,id_student,code_module,code_presentation,final_result,login_weekly,dominant_login_time,video_completion_rate,days_to_first_activity,forum_posts_count,submission_timeliness_days,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,date_registration,date_unregistration,vle_total_clicks,vle_active_days,vle_distinct_sites,vle_clicks_dataplus,vle_clicks_dualpane,vle_clicks_externalquiz,vle_clicks_folder,vle_clicks_forumng,vle_clicks_glossary,vle_clicks_homepage,vle_clicks_htmlactivity,vle_clicks_oucollaborate,vle_clicks_oucontent,vle_clicks_ouelluminate,vle_clicks_ouwiki,vle_clicks_page,vle_clicks_questionnaire,vle_clicks_quiz,vle_clicks_repeatactivity,vle_clicks_resource,vle_clicks_sharedsubpage,vle_clicks_subpage,vle_clicks_url,assess_n_submitted,assess_mean_score,assess_n_banked,assess_mean_days_before_deadline,assess_n_late,never_started,days_to_first_filled,missed_submission,late_submitter,prompt_starter,early_start_score,frequency_score,intensity_score,forum_per_login,completion_x_frequency,procrastination_score,low_engagement_flag,engagement_breadth,registration_delay_days,registered_before_start,is_unregistered,log_vle_total_clicks,log_vle_active_days,log_vle_distinct_sites,clicks_per_active_day,log_clicks_per_active_day,vle_clicks_dataplus_ratio,vle_clicks_dualpane_ratio,vle_clicks_externalquiz_ratio,vle_clicks_folder_ratio,vle_clicks_forumng_ratio,vle_clicks_glossary_ratio,vle_clicks_homepage_ratio,vle_clicks_htmlactivity_ratio,vle_clicks_oucollaborate_ratio,vle_clicks_oucontent_ratio,vle_clicks_ouelluminate_ratio,vle_clicks_ouwiki_ratio,vle_clicks_page_ratio,vle_clicks_questionnaire_ratio,vle_clicks_quiz_ratio,vle_clicks_repeatactivity_ratio,vle_clicks_resource_ratio,vle_clicks_sharedsubpage_ratio,vle_clicks_subpage_ratio,vle_clicks_url_ratio,vle_activity_type_diversity,assessment_completion_rate,late_assessment_rate,assignment_timeliness_score,login_time_Afternoon,login_time_Evening,login_time_Morning,login_time_Night
0,11391,AAA,2013J,Pass,2,Afternoon,0.934,9.0,5,-0.2,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,-159.0,NaN,934.0,40.0,55.0,0.0,0.0,0.0,0.0,193.0,0.0,138.0,0.0,0.0,553.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0,0.0,32.0,5.0,5.0,82.0,0.0,1.8,0.0,0,9.0,0,0,0,0.100000,1.239393,1.537005,1.666667,1.868,-0.597998,0,4,-159.0,1,0,6.840547,3.713572,4.025352,23.350000,3.192532,0.000000,0.0,0.0,0.0,0.206638,0.000000,0.147752,0.0,0.0,0.592077,0.0,0.0,0.0,0.0,0.0,0.0,0.013919,0.0,0.034261,0.005353,6,1.0,0.0,-0.319689,1,0,0,0
1,28400,AAA,2013J,Pass,6,Evening,0.879,2.0,0,-1.5,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,-53.0,NaN,1435.0,80.0,84.0,10.0,0.0,0.0,0.0,417.0,0.0,324.0,0.0,0.0,537.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,0.0,87.0,48.0,5.0,66.4,0.0,0.0,2.0,0,2.0,0,0,1,0.333333,0.394069,1.794023,0.000000,5.274,-1.242339,0,3,-53.0,1,0,7.269617,4.394449,4.442651,17.937500,2.941144,0.006969,0.0,0.0,0.0,0.290592,0.000000,0.225784,0.0,0.0,0.374216,0.0,0.0,0.0,0.0,0.0,0.0,0.008362,0.0,0.060627,0.033449,7,1.0,0.4,-0.394787,0,1,0,0
2,30268,AAA,2013J,Withdrawn,0,Evening,0.522,NaN,0,3.4,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,-92.0,12.0,281.0,12.0,22.0,0.0,0.0,0.0,0.0,126.0,0.0,59.0,0.0,0.0,66.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,22.0,4.0,0.0,0.0,0.0,0.0,0.0,1,90.0,1,0,0,0.010989,-1.330395,-0.742471,0.000000,0.000,3.250662,0,1,-92.0,1,1,5.641907,2.564949,3.135494,23.416667,3.195266,0.000000,0.0,0.0,0.0,0.448399,0.000000,0.209964,0.0,0.0,0.234875,0.0,0.0,0.0,0.0,0.0,0.0,0.014235,0.0,0.078292,0.014235,6,0.0,0.0,-0.394787,0,1,0,0
3,31604,AAA,2013J,Pass,8,Evening,0.521,3.0,2,2.2,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,-52.0,NaN,2158.0,123.0,82.0,2.0,0.0,0.0,0.0,634.0,1.0,432.0,0.0,0.0,836.0,0.0,0.0,0.0,0.0,0.0,0.0,19.0,0.0,144.0,90.0,5.0,76.0,0.0,2.0,0.0,0,3.0,0,0,1,0.250000,1.766877,-0.366957,0.222222,4.168,-0.042435,0,3,-52.0,1,0,7.677400,4.820282,4.418841,17.544715,2.920185,0.000927,0.0,0.0,0.0,0.293791,0.000463,0.200185,0.0,0.0,0.387396,0

Feature table shape: (32593, 97)


In [5]:
base_behavior_features = [
    "login_weekly",
    "video_completion_rate",
    "forum_posts_count",
    "days_to_first_filled",
    "submission_timeliness_days",
    "never_started",
    "missed_submission",
    "late_submitter",
    "prompt_starter",
    "low_engagement_flag",
    "early_start_score",
    "frequency_score",
    "intensity_score",
    "forum_per_login",
    "completion_x_frequency",
    "procrastination_score",
    "engagement_breadth",
    "registration_delay_days",
    "registered_before_start",
]

vle_features = first_existing(feature_df.columns, [
    "vle_total_clicks",
    "vle_active_days",
    "vle_distinct_sites",
    "log_vle_total_clicks",
    "log_vle_active_days",
    "log_vle_distinct_sites",
    "clicks_per_active_day",
    "log_clicks_per_active_day",
    "vle_activity_type_diversity",
])

vle_ratio_features = [
    c for c in feature_df.columns
    if c.startswith("vle_clicks_") and c.endswith("_ratio")
]

preferred_vle_ratios = first_existing(feature_df.columns, [
    "vle_clicks_forumng_ratio",
    "vle_clicks_resource_ratio",
    "vle_clicks_oucontent_ratio",
    "vle_clicks_quiz_ratio",
    "vle_clicks_homepage_ratio",
    "vle_clicks_subpage_ratio",
    "vle_clicks_url_ratio",
])

assessment_features = first_existing(feature_df.columns, [
    "assess_n_submitted",
    "assess_mean_score",
    "assess_n_banked",
    "assess_mean_days_before_deadline",
    "assess_n_late",
    "assessment_completion_rate",
    "late_assessment_rate",
    "assignment_timeliness_score",
])

login_time_features = [c for c in feature_df.columns if c.startswith("login_time_")]

candidate_features = []
for group in [
    base_behavior_features,
    vle_features,
    preferred_vle_ratios,
    assessment_features,
    login_time_features,
]:
    for col in group:
        if col in feature_df.columns and col not in candidate_features:
            candidate_features.append(col)

feature_df[candidate_features] = feature_df[candidate_features].replace([np.inf, -np.inf], np.nan)
for col in candidate_features:
    feature_df[col] = feature_df[col].fillna(feature_df[col].median() if pd.api.types.is_numeric_dtype(feature_df[col]) else 0)

raw_candidate_path = OUTPUT_DIR / "engineered_features_raw.csv"
feature_df[KEY_COLS + candidate_features + ["final_result"]].to_csv(raw_candidate_path, index=False)

feature_overview = pd.DataFrame({
    "feature": candidate_features,
    "mean": feature_df[candidate_features].mean().round(4).values,
    "std": feature_df[candidate_features].std().round(4).values,
    "missing_pct": feature_df[candidate_features].isna().mean().mul(100).round(2).values,
    "n_unique": feature_df[candidate_features].nunique().values,
})
display(feature_overview)
print("Candidate features:", len(candidate_features))
print("Saved:", raw_candidate_path)

,feature,mean,std,missing_pct,n_unique
0,login_weekly,2.6166,3.4793,0.0,25
1,video_completion_rate,0.4445,0.3600,0.0,1001
2,forum_posts_count,1.4495,2.5063,0.0,18
3,days_to_first_filled,21.5603,29.8476,0.0,91
4,submission_timeliness_days,0.3621,3.1722,0.0,222
5,never_started,0.1391,0.3461,0.0,2
6,missed_submission,0.2160,0.4115,0.0,2
7,late_submitter,0.0167,0.1280,0.0,2
8,prompt_starter,0.5135,0.4998,0.0,2
9,low_engagement_flag,0.2178,0.4128,0.0,2


Candidate features: 47
Saved: artifacts/engineered_features_raw.csv


## Downstream handoff

Current project lineage:

`feature engineering output` → group recleaning step → `data_reclean_after_fe.csv` → `data_reconcile.ipynb`

The recleaning code belongs to the other agreed team workflow and is intentionally not reproduced here.
